In [ ]:
import pandas as pd
import json
import os
from pathlib import Path

In [ ]:
current_directory= Path.cwd()
project_root=current_directory.parent
actual_directory=project_root/"data"/"raw"/"datos_sucios_hito1.csv"
df_territories_city=pd.read_csv(actual_directory)
df_territories_city

In [ ]:
city_list=df_territories_city["Municipio"].unique().tolist()
city_list

In [ ]:
df_territories_accent=df_territories_city[df_territories_city["Región"].str.contains("á|é|í|ó|ú",case=False)]
list_accent=df_territories_accent["Región"].unique().tolist()
list_accent

In [ ]:
def accent_normalization(text):
    '''
    Elimina acentos ortográficos del input.

    Args:
        text (str): cadena de texto.

    Returns:
        str: cadena de texto sin acentos ortográficos.
    '''
    return (text.lower().strip()
            .replace("á","a")
            .replace("é","e")
            .replace("í","i")
            .replace("ó","o")
            .replace("ú","u"))
accent_normalization_dic={accent_normalization(a): a for a in list_accent} #Crea un diccionario con la llave->región sin tilde, espacios en los extremos y en minúscula y el valor->región intacto desde la lista original.

def is_input_valid(text):
    '''
    Devuelve un valor boleano true/false según si el input cumple con las condiciones de una cadena de texto según los parámetros de unicode.

    Args:
        text (str): cadena de texto.

    Returns:
        bool: false si el argumento es vacío, 0 o none. Si no es vacío, 0 o none, devolverá true siempre que esté compuesto únicamente por letras (según la categoría 'Letter' de Unicode), incluso si la cadena de texto tiene espacios internos, gracias al uso de replace().
    
    Raises:
        AttributeError: si se pasa un valor sin método .replace() (ej. un int).
    '''
    if not text:
        return False
    return text.replace(" ","").isalpha() #El AttributeError que se menciona no puede ocurrir en el flujo actual, pues a esta función solo llegarán valores tipo str.

In [ ]:
progress_root=project_root/"data"/"interim"
progress_root.mkdir(parents=True,exist_ok=True)
progress_file=progress_root/"progress_territories.json"

if progress_file.exists():
    with open(progress_file, "r") as f: #Se descartó pd.read_json(..., typ='series') para cargar el mapeo de regiones, porque el archivo es técnicamente un diccionario, no un DataFrame — pandas interpretaría las llaves como columnas y los valores como filas, sin índices reales. Es más correcto y directo usar la librería json estándar.
        raw_data_saved=json.load(f)
else:
    raw_data_saved={}
data_saved={k.strip():v for k,v in raw_data_saved.items()}

In [ ]:
def city_territories(city_list,progress=None):
    '''
    Genera un diccionario con cada llave->municipio y su valor->región asignado.

    Args:

        city_list (list): contiene cada municipio que se encuentra en el DataFrame original.
        progress (dict, optional): diccionario que contiene cada llave->municipio y su valor->región asignado. Defaults to None.

    Returns:
        dict: diccionario con cada llave->municipio y su respectivo valor->región asignado.
    
    Raises:
        NameError: si accent_normalization_dic no está definido desde antes.

    Notes:
        Esta función depende de 'accent_normalization_dic', un diccionario que se genera en una función previa. Si no está definido antes de ejecutar esta función
        generará un NameError.

        Para mantener la modularidad, esta función no genera el diccionario 'progress' por sí misma. El diccionario con el que trabaja este argumento viene de afuera, si existe y, si no, lo asume como None.
    '''

    if progress is None:
        city_territories_dic={}
    else:
        city_territories_dic=progress.copy()
    
    for c in city_list:

        if c in city_territories_dic:
            continue

        while True:
            territories_raw=input(f"Ingrese una región para {c}: \n O ingrese '-' para terminar.").strip()
            
            if territories_raw=="-":
                return city_territories_dic
            
            if is_input_valid(territories_raw):
                break
            print ("La región solo puede contener letras.")

        territories_key=accent_normalization(territories_raw)
        territories_processed=accent_normalization_dic.get(territories_key,territories_raw.title())


        print(f"Municipio: {c}, región: {territories_processed}")
        city_territories_dic[c]=territories_processed

    return city_territories_dic

rpoint=city_territories(city_list, progress=data_saved)
with open (progress_file,"w") as f:
    json.dump(rpoint,f)

print(json.dumps(rpoint, indent=4, ensure_ascii=False))

In [ ]:
df_territories_city["Municipio"]=df_territories_city["Municipio"].str.strip() #Se normalizan los espacios en Municipio con .str.strip() porque el CSV original trae inconsistencias de espacios que impiden que las llaves coincidan exactamente con las del JSON de regiones. Sin esto, el .map() posterior dejaría varias filas sin región asignada.
df_territories_city

In [ ]:
with open(progress_file,"r") as f:
    mapping_dict=json.load(f)
print(mapping_dict)

In [ ]:
df_territories_city["Región"]=df_territories_city["Municipio"].map(mapping_dict) #Se cruza el diccionario con la tabla original. El diccionario tiene como llave al municipio y la región es el valor. Para buscar en el archivo original las llaves y reemplazar sus valores (cruce con el diccionario). Se usa map en vez de np.where, pues no hay condiciones lógicas que permitan fácilmente la comparación, sino una referencia 1 a 1; np.where es útil para cuando hay condiciones lógicas.
df_territories_city

In [ ]:
saved_output_directory=project_root/"data"/"processed"
saved_output_directory.mkdir(parents=True, exist_ok=True)
saved_file=saved_output_directory/"datos_limpios_hito1.csv"

df_territories_city.to_csv(saved_file,index=False,encoding="utf-8") #Se guarda la versión corregida: si se quisiera sobreescribr, basta con escribir el mismo nombre.